# FEniCSx, five PDE workflows


## 0. Setup

$$
\text{mesh}\rightarrow \text{space}\rightarrow \text{BCs}\rightarrow
\text{weak form}\rightarrow \text{PETSc solve}\rightarrow \text{logs and plots}.
$$


In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import time
import shutil
import subprocess
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import ufl

from basix.ufl import element, mixed_element
from dolfinx import fem, mesh, plot
from dolfinx.fem import petsc as fem_petsc
from scipy import sparse

try:
    import dolfinx_mpc
except ImportError:
    dolfinx_mpc = None

comm = MPI.COMM_WORLD
rank = comm.rank

SOLUTIONS_DIR = Path("solutions")
SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)


# Apply a portable Matplotlib style for all notebook figures.
def set_plot_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans", "sans-serif"],
        "mathtext.fontset": "dejavusans",
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "figure.titlesize": 11,
        "axes.linewidth": 0.8,
        "grid.linewidth": 0.5,
        "lines.linewidth": 1.4,
        "lines.markersize": 4,
        "figure.dpi": 120,
        "savefig.dpi": 220,
        "savefig.bbox": "tight",
        "savefig.facecolor": "white",
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    })

try:
    pv.set_jupyter_backend("static") # trame
except Exception:
    pass

try:
    pv.start_xvfb()
except Exception:
    pass


# Print only once in MPI runs.
def log(msg=""):
    if rank == 0:
        print(msg, flush=True)


# Print a clean section banner for notebook execution logs.
def banner(title):
    log("")
    log("=" * 78)
    log(title)
    log("=" * 78)


# Create a tetrahedral unit-cube mesh with a readable name.
def make_cube(n=10, name="Omega"):
    domain = mesh.create_unit_cube(comm, n, n, n, cell_type=mesh.CellType.tetrahedron)
    domain.name = name
    return domain


# Find boundary facets satisfying a coordinate predicate.
def cube_facets(domain, marker):
    fdim = domain.topology.dim - 1 # = 2
    return mesh.locate_entities_boundary(domain, fdim, marker)


# Mark every exterior face of the unit cube.
def all_boundary(x):
    return (
        np.isclose(x[0], 0.0)
        | np.isclose(x[0], 1.0)
        | np.isclose(x[1], 0.0)
        | np.isclose(x[1], 1.0)
        | np.isclose(x[2], 0.0)
        | np.isclose(x[2], 1.0)
    )


# Build a scalar continuous Lagrange function space.
def scalar_space(domain, degree=1):
    return fem.functionspace(domain, ("Lagrange", degree))


# Build a vector-valued continuous Lagrange function space.
def vector_space(domain, degree=1):
    return fem.functionspace(domain, ("Lagrange", degree, (domain.geometry.dim,)))


# Count global algebraic DOFs including block size.
def num_dofs(V):
    return V.dofmap.index_map.size_global * V.dofmap.index_map_bs


# Compute the maximum local cell diameter h on the mesh.
def hmax(domain):
    tdim = domain.topology.dim
    domain.topology.create_connectivity(tdim, 0)
    c_to_v = domain.topology.connectivity(tdim, 0)
    coords = domain.geometry.x
    local_h = 0.0
    for cell in range(domain.topology.index_map(tdim).size_local):
        vertices = c_to_v.links(cell)
        X = coords[vertices]
        for i in range(len(X)):
            for j in range(i + 1, len(X)):
                local_h = max(local_h, float(np.linalg.norm(X[i] - X[j])))
    return comm.allreduce(local_h, op=MPI.MAX)


# Print the starting mesh and space size for a problem.
def problem_start(label, domain, V):
    log(f"[{label}] mesh cells    : {domain.topology.index_map(domain.topology.dim).size_global}")
    log(f"[{label}] dofs          : {num_dofs(V)}")
    log(f"[{label}] h_max         : {hmax(domain):.6e}")


# PETSc options for robust direct solves in small teaching examples.
def direct_lu_options():
    return {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}


# PETSc options for moderate nonsymmetric systems.
def gmres_lu_options():
    return {
        "ksp_type": "gmres",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "ksp_rtol": 1e-10,
        "ksp_max_it": 200,
    }


# Extract and print PETSc KSP status after a LinearProblem solve.
def report_ksp(problem, label):
    solver = getattr(problem, "solver", None)
    if solver is None:
        log(f"[{label}] PETSc KSP     : unavailable")
        return
    reason = solver.getConvergedReason()
    its = solver.getIterationNumber()
    log(f"[{label}] PETSc KSP     : reason={reason}, iterations={its}")


# Time a PETSc-backed solve call and report the elapsed wall time.
def timed_solve(label, solve_callable):
    log(f"[{label}] assembly/solve : start")
    t0 = time.perf_counter()
    result = solve_callable()
    elapsed = time.perf_counter() - t0
    log(f"[{label}] assembly/solve : done in {elapsed:.4f} s")
    return result, elapsed


# Assemble an L2 norm from a UFL expression.
def l2_norm(expr, domain):
    local = fem.assemble_scalar(fem.form(expr * ufl.dx(domain)))
    return np.sqrt(comm.allreduce(local, op=MPI.SUM))


# Project a scalar or vector expression to a visualization space.
def project(expr, V, bcs=(), options=None):
    out = fem.Function(V)
    trial = ufl.TrialFunction(V)
    test = ufl.TestFunction(V)
    a = ufl.inner(trial, test) * ufl.dx
    L = ufl.inner(expr, test) * ufl.dx
    problem = fem_petsc.LinearProblem(a, L, bcs=list(bcs), u=out, petsc_options=options or direct_lu_options())
    problem.solve()
    out.x.scatter_forward()
    return out


# Convert a mesh into a PyVista unstructured grid.
def mesh_grid(domain):
    tdim = domain.topology.dim
    domain.topology.create_connectivity(tdim, tdim)
    topology, cell_types, x = plot.vtk_mesh(domain, tdim)
    return pv.UnstructuredGrid(topology, cell_types, x)


# Convert a scalar or vector function space into a PyVista grid.
def function_grid(u):
    topology, cell_types, x = plot.vtk_mesh(u.function_space)
    return pv.UnstructuredGrid(topology, cell_types, x)


# Write a PyVista mesh or solution grid to the local solutions directory.
def save_pyvista_grid(grid, filename):
    if rank != 0:
        return None
    path = SOLUTIONS_DIR / filename
    grid.save(path)
    log(f"[export] 3D dataset   : {path}")
    return path


# Save the volume mesh as a ParaView-readable VTU file.
def save_mesh_vtu(domain, filename):
    return save_pyvista_grid(mesh_grid(domain), filename)


# Save a scalar finite-element function as a ParaView-readable VTU file.
def save_scalar_vtu(u, filename, field="u"):
    return save_pyvista_grid(scalar_grid(u, field), filename)


# Save a scalar z=0.5 slice as a ParaView-readable VTP file.
def save_scalar_slice_vtp(u, filename, field="u"):
    if rank != 0:
        return None
    grid = scalar_grid(u, field)
    slc = grid.slice(normal="z", origin=(0.5, 0.5, 0.5))
    return save_pyvista_grid(slc, filename)


# Save a vector finite-element function as a ParaView-readable VTU file.
def save_vector_vtu(q, filename, field="q"):
    if rank != 0:
        return None
    gdim = q.function_space.mesh.geometry.dim
    grid = function_grid(q)
    values = np.asarray(q.x.array.real).reshape((-1, gdim))
    grid.point_data[field] = values
    grid.point_data[f"|{field}|"] = np.linalg.norm(values, axis=1)
    return save_pyvista_grid(grid, filename)


# Save small residual/history arrays as CSV files.
def save_history_csv(filename, header, rows):
    if rank != 0:
        return None
    path = SOLUTIONS_DIR / filename
    np.savetxt(path, np.asarray(list(rows), dtype=float), delimiter=",", header=",".join(header), comments="")
    log(f"[export] history      : {path}")
    return path


# Show the cube as a 3D surface with visible mesh edges.
def plot_mesh_surface_edges(domain, title):
    if rank != 0:
        return
    grid = mesh_grid(domain)
    surf = grid.extract_surface()
    p = pv.Plotter(window_size=(720, 520))
    p.add_mesh(surf, color="#d9d9d9", show_edges=True, edge_color="black", opacity=0.42)
    p.add_axes()
    p.add_title(title, font_size=10)
    p.show(jupyter_backend="static")


# Attach scalar values to a PyVista grid for scalar plotting.
def scalar_grid(u, field="u"):
    grid = function_grid(u)
    values = np.asarray(u.x.array.real)
    if values.size == grid.n_points:
        grid.point_data[field] = values
    else:
        grid.cell_data[field] = values[: grid.n_cells]
    return grid


# Show scalar data on the outer 3D surface with mesh edges.
def plot_scalar_surface_edges(u, title, field="u", cmap="viridis"):
    if rank != 0:
        return
    grid = scalar_grid(u, field)
    surf = grid.extract_surface()
    p = pv.Plotter(window_size=(760, 560))
    p.add_mesh(surf, scalars=field, cmap=cmap, show_edges=True, edge_color="black")
    p.add_axes()
    p.add_title(title, font_size=10)
    p.show(jupyter_backend="static")


# Show scalar data on an interior slice plus the cube outline.
def plot_scalar_slice(u, title, field="u", cmap="viridis"):
    if rank != 0:
        return
    grid = scalar_grid(u, field)
    slc = grid.slice(normal="z", origin=(0.5, 0.5, 0.5))
    p = pv.Plotter(window_size=(760, 560))
    p.add_mesh(slc, scalars=field, cmap=cmap, show_edges=True, edge_color="white")
    p.add_mesh(grid.outline(), color="black")
    p.add_axes()
    p.add_title(title, font_size=10)
    p.show(jupyter_backend="static")


# Show a vector field as glyphs over a colored slice.
def plot_vector_slice(q, title, field="q"):
    if rank != 0:
        return
    gdim = q.function_space.mesh.geometry.dim
    grid = function_grid(q)
    values = np.asarray(q.x.array.real).reshape((-1, gdim))
    grid.point_data[field] = values
    grid.point_data[f"|{field}|"] = np.linalg.norm(values, axis=1)
    slc = grid.slice(normal="z", origin=(0.5, 0.5, 0.5))
    glyphs = slc.glyph(orient=field, scale=f"|{field}|", factor=0.10)
    p = pv.Plotter(window_size=(760, 560))
    p.add_mesh(slc, scalars=f"|{field}|", cmap="magma", opacity=0.70)
    p.add_mesh(glyphs, color="black")
    p.add_mesh(grid.outline(), color="black")
    p.add_axes()
    p.add_title(title, font_size=10)
    p.show(jupyter_backend="static")


# Plot a small convergence or time-history curve.
def plot_history(history, title, ylabel, yscale=None):
    if rank != 0:
        return
    set_plot_style()
    fig, ax = plt.subplots(figsize=(6.0, 3.4))
    ax.plot(np.arange(len(history)), history, marker="o", linewidth=1.4)
    ax.set_xlabel("step")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.30)
    if yscale:
        ax.set_yscale(yscale)
    plt.show()



# Assemble a PETSc matrix from a bilinear form and optional Dirichlet conditions.
def assemble_form_matrix(a_form, bcs=()):
    A = fem_petsc.assemble_matrix(fem.form(a_form), bcs=list(bcs))
    A.assemble()
    return A


# Assemble the constrained matrix for an MPC bilinear form.
def assemble_mpc_matrix(a_form, mpc, bcs=()):
    A = dolfinx_mpc.assemble_matrix(fem.form(a_form), mpc, bcs=list(bcs))
    A.assemble()
    return A


# Convert a PETSc AIJ matrix into a SciPy CSR matrix for plotting.
def petsc_to_csr(A):
    indptr, indices, data = A.getValuesCSR()
    return sparse.csr_matrix((data, indices, indptr), shape=A.getSize())


# Plot the sparsity pattern of an assembled matrix with the shared manuscript style.
def plot_sparsity(A, title, max_points=90000):
    if rank != 0:
        return
    set_plot_style()
    S = petsc_to_csr(A)
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    marker_size = max(0.1, min(1.0, max_points / max(S.nnz, 1)))
    ax.spy(S, markersize=marker_size, color="black")
    ax.set_title(f"{title}\nshape={S.shape}, nnz={S.nnz}")
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    plt.show()


# Plot and export a time-dependent z=0.5 slice animation for scalar heat snapshots.
def export_heat_slice_animation(snapshots, out_dir=SOLUTIONS_DIR, stem="heat_slice_evolution"):
    if rank != 0:
        return None
    set_plot_style()
    out_dir.mkdir(exist_ok=True)
    times = sorted(snapshots)
    if not times:
        return None
    first = snapshots[times[0]]
    coords = first.function_space.tabulate_dof_coordinates()
    z_mask = np.isclose(coords[:, 2], 0.5)
    if not np.any(z_mask):
        z_values = np.unique(np.round(coords[:, 2], 12))
        z0 = z_values[np.argmin(np.abs(z_values - 0.5))]
        z_mask = np.isclose(coords[:, 2], z0)
    X = coords[z_mask, 0]
    Y = coords[z_mask, 1]
    order = np.lexsort((X, Y))
    X = X[order]
    Y = Y[order]
    frame_arrays = [np.asarray(snapshots[t].x.array.real)[z_mask][order] for t in times]
    vmin = min(float(np.min(v)) for v in frame_arrays)
    vmax = max(float(np.max(v)) for v in frame_arrays)
    frames = []
    for t_value, values in zip(times, frame_arrays):
        fig, ax = plt.subplots(figsize=(5.2, 4.2))
        tri = ax.tricontourf(X, Y, values, levels=24, cmap="viridis", vmin=vmin, vmax=vmax)
        ax.tricontour(X, Y, values, levels=8, colors="black", linewidths=0.25, alpha=0.35)
        ax.set_aspect("equal")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title(f"Heat slice z=0.5, t={t_value:.2f}")
        fig.colorbar(tri, ax=ax, shrink=0.82, label="u")
        fig.canvas.draw()
        rgba = np.asarray(fig.canvas.buffer_rgba())
        frames.append(rgba[:, :, :3].copy())
        plt.close(fig)
    mp4_path = out_dir / f"{stem}.mp4"
    gif_path = out_dir / f"{stem}.gif"
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        frame_dir = out_dir / f"{stem}_frames"
        frame_dir.mkdir(exist_ok=True)
        for j, frame in enumerate(frames):
            imageio.imwrite(frame_dir / f"frame_{j:04d}.png", frame)
        cmd = [
            ffmpeg,
            "-y",
            "-framerate",
            "4",
            "-i",
            str(frame_dir / "frame_%04d.png"),
            "-pix_fmt",
            "yuv420p",
            "-vcodec",
            "libx264",
            str(mp4_path),
        ]
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            log(f"[Heat] animation    : {mp4_path}")
            return mp4_path
        except Exception as exc:
            log(f"[Heat] animation mp4 failed, falling back to GIF: {exc}")
    imageio.mimsave(gif_path, frames, fps=4)
    log(f"[Heat] animation    : {gif_path}")
    return gif_path


## 1. Advection-diffusion-reaction

$$
-\nabla\cdot(\nabla u)+\mathbf b\cdot\nabla u+cu=f,
\qquad \mathbf b=(1,0.5,0.25),\qquad c=2.
$$

Manufactured data:

$$
u_e=x(1-x)y(1-y)z(1-z),
\qquad
f=-\Delta u_e+\mathbf b\cdot\nabla u_e+c u_e.
$$

Weak form in $V_h=\mathrm{CG}_1$:

$$
\int_\Omega \nabla u\cdot\nabla v
+\int_\Omega(\mathbf b\cdot\nabla u)v
+\int_\Omega c\,uv
=\int_\Omega fv.
$$


In [ ]:
banner("1. Advection-diffusion-reaction")
domain = make_cube(12, "ADR cube")
plot_mesh_surface_edges(domain, "ADR cube mesh: 3D surface with edges")
save_mesh_vtu(domain, "adr_mesh.vtu")

V = scalar_space(domain, 1)
problem_start("ADR", domain, V)
x = ufl.SpatialCoordinate(domain)
u_exact_expr = x[0] * (1 - x[0]) * x[1] * (1 - x[1]) * x[2] * (1 - x[2])
b = ufl.as_vector((1.0, 0.5, 0.25))
c = fem.Constant(domain, PETSc.ScalarType(2.0))
f = -ufl.div(ufl.grad(u_exact_expr)) + ufl.dot(b, ufl.grad(u_exact_expr)) + c * u_exact_expr

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
a = (ufl.dot(ufl.grad(u), ufl.grad(v)) + ufl.dot(b, ufl.grad(u)) * v + c * u * v) * ufl.dx
L = f * v * ufl.dx

u_D = fem.Function(V, name="u_exact")
u_D.interpolate(lambda X: X[0] * (1 - X[0]) * X[1] * (1 - X[1]) * X[2] * (1 - X[2]))
facets = cube_facets(domain, all_boundary)
dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, facets)
bc = fem.dirichletbc(u_D, dofs)

uh = fem.Function(V, name="u_adr")
A_adr = assemble_form_matrix(a, [bc])
problem = fem_petsc.LinearProblem(a, L, bcs=[bc], u=uh, petsc_options=direct_lu_options())
_, solve_time = timed_solve("ADR", problem.solve)
uh.x.scatter_forward()
report_ksp(problem, "ADR")

u_error = fem.Function(V, name="u_h_minus_u_e")
u_error.x.array[:] = uh.x.array - u_D.x.array
err_l2 = l2_norm((uh - u_D) ** 2, domain)
log(f"[ADR] L2 error     : {err_l2:.6e}")
log(f"[ADR] solve timing : {solve_time:.4f} s")

save_scalar_vtu(uh, "adr_solution.vtu", "u_h")
save_scalar_vtu(u_error, "adr_error.vtu", "u_h_minus_u_e")
save_scalar_slice_vtp(uh, "adr_solution_zslice.vtp", "u_h")
plot_scalar_surface_edges(uh, "ADR solution on 3D cube surface", "u_h")
plot_scalar_slice(uh, "ADR solution, z=0.5 slice", "u_h")
plot_scalar_slice(u_error, "ADR error, z=0.5 slice", "u_h-u_e", cmap="coolwarm")


In [ ]:
plot_sparsity(A_adr, "ADR stiffness/advection matrix")


## 2. Mixed Darcy flow

$$
\mathbf q=-\nabla p,\qquad \nabla\cdot\mathbf q=f.
$$

Boundary data:

$$
p=1 \text{ on } x=0,\qquad p=0 \text{ on } x=1,\qquad
\mathbf q\cdot\mathbf n=0 \text{ on } y,z \text{ walls}.
$$

Mixed weak form in $RT_1\times DG_0$:

$$
\int_\Omega \mathbf q\cdot\boldsymbol\tau
-\int_\Omega p\,\nabla\cdot\boldsymbol\tau
+\int_\Omega (\nabla\cdot\mathbf q)w
=-\int_{\Gamma_p} p_D\,\boldsymbol\tau\cdot\mathbf n+\int_\Omega f w.
$$


In [ ]:
banner("2. Mixed Darcy flow")
domain = make_cube(10, "Darcy cube")
plot_mesh_surface_edges(domain, "Darcy cube mesh: 3D surface with edges")
save_mesh_vtu(domain, "darcy_mesh.vtu")

cell = domain.basix_cell()
RT = element("RT", cell, 1)
DG0 = element("DG", cell, 0)
W = fem.functionspace(domain, mixed_element([RT, DG0]))
problem_start("Darcy", domain, W)

q, p = ufl.TrialFunctions(W)
tau, w = ufl.TestFunctions(W)
x = ufl.SpatialCoordinate(domain)
n = ufl.FacetNormal(domain)
f = 8.0 * ufl.sin(np.pi * x[0]) * ufl.sin(np.pi * x[1]) * ufl.sin(np.pi * x[2])

fdim = domain.topology.dim - 1
left_facets = cube_facets(domain, lambda X: np.isclose(X[0], 0.0))
right_facets = cube_facets(domain, lambda X: np.isclose(X[0], 1.0))
noflux_facets = cube_facets(
    domain,
    lambda X: np.isclose(X[1], 0.0)
    | np.isclose(X[1], 1.0)
    | np.isclose(X[2], 0.0)
    | np.isclose(X[2], 1.0),
)
facet_indices = np.hstack([left_facets, right_facets]).astype(np.int32)
facet_markers = np.hstack([
    np.full(len(left_facets), 1, dtype=np.int32),
    np.full(len(right_facets), 2, dtype=np.int32),
])
facet_order = np.argsort(facet_indices)
tags = mesh.meshtags(domain, fdim, facet_indices[facet_order], facet_markers[facet_order])
ds = ufl.Measure("ds", domain=domain, subdomain_data=tags)

Q0, _ = W.sub(0).collapse()
q_zero = fem.Function(Q0)
q_dofs = fem.locate_dofs_topological((W.sub(0), Q0), fdim, noflux_facets)
bc_qn = fem.dirichletbc(q_zero, q_dofs, W.sub(0))

a = (
    ufl.inner(q, tau) * ufl.dx
    - p * ufl.div(tau) * ufl.dx
    + ufl.div(q) * w * ufl.dx
)
L = f * w * ufl.dx - 1.0 * ufl.dot(tau, n) * ds(1) - 0.0 * ufl.dot(tau, n) * ds(2)

wh = fem.Function(W, name="darcy_state")
A_darcy = assemble_form_matrix(a, [bc_qn])
problem = fem_petsc.LinearProblem(a, L, bcs=[bc_qn], u=wh, petsc_options=direct_lu_options())
_, solve_time = timed_solve("Darcy", problem.solve)
wh.x.scatter_forward()
report_ksp(problem, "Darcy")

qh = wh.sub(0).collapse()
ph = wh.sub(1).collapse()
V1 = scalar_space(domain, 1)
Vv = vector_space(domain, 1)
p_vis = project(ph, V1)
q_vis = project(qh, Vv)
q_mag = project(ufl.sqrt(ufl.inner(qh, qh)), V1)
mean_p = fem.assemble_scalar(fem.form(ph * ufl.dx))
log(f"[Darcy] mean pressure: {comm.allreduce(mean_p, op=MPI.SUM):.6e}")
log(f"[Darcy] solve timing : {solve_time:.4f} s")

save_scalar_vtu(p_vis, "darcy_pressure.vtu", "p")
save_scalar_vtu(q_mag, "darcy_velocity_magnitude.vtu", "|q|")
save_vector_vtu(q_vis, "darcy_flux.vtu", "q")
save_scalar_slice_vtp(p_vis, "darcy_pressure_zslice.vtp", "p")
plot_scalar_surface_edges(p_vis, "Darcy pressure on 3D cube surface", "p")
plot_scalar_slice(p_vis, "Darcy pressure, z=0.5 slice", "p")
plot_scalar_surface_edges(q_mag, "Darcy velocity magnitude on 3D cube surface", "|q|", cmap="magma")
plot_vector_slice(q_vis, "Darcy flux q, z=0.5 slice", "q")


In [ ]:
plot_sparsity(A_darcy, "Darcy mixed saddle-point matrix")


## 3. Nonlinear p-Laplacian

$$
-\nabla\cdot\left(|\nabla u|^{p-2}\nabla u\right)=f,\qquad
p=4,\qquad u=0\text{ on }\partial\Omega.
$$

Residual and automatic Jacobian:

$$
F(u;v)=\int_\Omega (|\nabla u|^2+\varepsilon^2)^{(p-2)/2}
\nabla u\cdot\nabla v-\int_\Omega fv=0,\qquad
J=F'(u).
$$


In [ ]:
banner("3. Nonlinear p-Laplacian")
domain = make_cube(10, "p-Laplacian cube")
plot_mesh_surface_edges(domain, "p-Laplacian cube mesh: 3D surface with edges")
save_mesh_vtu(domain, "p_laplacian_mesh.vtu")

V = scalar_space(domain, 1)
problem_start("p-Laplacian", domain, V)
x = ufl.SpatialCoordinate(domain)
p_power = fem.Constant(domain, PETSc.ScalarType(4.0))
eps = fem.Constant(domain, PETSc.ScalarType(1.0e-4))
force = 1.0 * ufl.sin(np.pi * x[0]) * ufl.sin(np.pi * x[1]) * ufl.sin(np.pi * x[2])

u = fem.Function(V, name="u_p_laplace")
u.interpolate(lambda X: 0.2 * np.sin(np.pi * X[0]) * np.sin(np.pi * X[1]) * np.sin(np.pi * X[2]))
v = ufl.TestFunction(V)
grad_norm = ufl.sqrt(ufl.inner(ufl.grad(u), ufl.grad(u)) + eps**2)
F_form = grad_norm ** (p_power - 2.0) * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx - force * v * ufl.dx
J_form = ufl.derivative(F_form, u)

facets = cube_facets(domain, all_boundary)
dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, facets)
bc = fem.dirichletbc(PETSc.ScalarType(0.0), dofs, V)


# Wrap UFL residual/Jacobian forms for PETSc SNES callbacks.
class SNESProblem:
    # Store compiled residual and Jacobian forms plus the active solution vector.
    def __init__(self, residual, jacobian, solution, bcs):
        self.residual = fem.form(residual)
        self.jacobian = fem.form(jacobian)
        self.solution = solution
        self.bcs = bcs

    # Assemble the nonlinear residual vector for the current SNES iterate.
    def F(self, snes, x_vec, f_vec):
        x_vec.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)
        x_vec.copy(self.solution.x.petsc_vec)
        self.solution.x.scatter_forward()
        with f_vec.localForm() as local:
            local.set(0.0)
        fem_petsc.assemble_vector(f_vec, self.residual)
        fem_petsc.apply_lifting(f_vec, [self.jacobian], [self.bcs], x0=[x_vec], alpha=-1.0)
        f_vec.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        fem_petsc.set_bc(f_vec, self.bcs, x_vec, -1.0)

    # Assemble the Newton Jacobian matrix for the current SNES iterate.
    def J(self, snes, x_vec, A, P):
        A.zeroEntries()
        fem_petsc.assemble_matrix(A, self.jacobian, bcs=self.bcs)
        A.assemble()


A_plaplace_jacobian = assemble_form_matrix(J_form, [bc])

nl = SNESProblem(F_form, J_form, u, [bc])
b_vec = fem_petsc.create_vector(nl.residual)
A = fem_petsc.create_matrix(nl.jacobian)

snes = PETSc.SNES().create(comm)
snes.setFunction(nl.F, b_vec)
snes.setJacobian(nl.J, A)
snes.setType("newtonls")
snes.setTolerances(atol=1e-10, rtol=1e-8, max_it=25)
ksp = snes.getKSP()
ksp.setType("preonly")
ksp.getPC().setType("lu")
snes.setFromOptions()

residual_history = []


# Print one aligned Newton residual line per SNES iteration.
def monitor(snes, its, norm):
    residual_history.append(norm)
    log(f"[p-Laplacian] Newton {its:02d}  residual={norm: .6e}")


snes.setMonitor(monitor)
_, solve_time = timed_solve("p-Laplacian", lambda: snes.solve(None, u.x.petsc_vec))
u.x.scatter_forward()

log(f"[p-Laplacian] SNES reason  : {snes.getConvergedReason()}")
log(f"[p-Laplacian] Newton its   : {snes.getIterationNumber()}")
log(f"[p-Laplacian] solve timing : {solve_time:.4f} s")
save_history_csv("p_laplacian_residuals.csv", ["iteration", "residual"], enumerate(residual_history))
save_scalar_vtu(u, "p_laplacian_solution.vtu", "u")
save_scalar_slice_vtp(u, "p_laplacian_solution_zslice.vtp", "u")
plot_scalar_surface_edges(u, "p-Laplacian solution on 3D cube surface", "u")
plot_scalar_slice(u, "p-Laplacian solution, z=0.5 slice", "u")
plot_history(residual_history, "SNES Newton residual history", r"$\|F(u_k)\|_2$", yscale="log")


In [ ]:
plot_sparsity(A_plaplace_jacobian, "p-Laplacian Newton Jacobian at initial guess")


## 4. Time-dependent heat equation

$$
u_t-\Delta u=f(x,y,z,t),\qquad
f=\sin(\pi x)\sin(\pi y)\sin(\pi z)e^{-t},\qquad
u(\cdot,0)=0.
$$

Implicit Euler:

$$
\frac{u^{n+1}-u^n}{\Delta t}-\Delta u^{n+1}=f^{n+1}.
$$

Weak form:

$$
\int_\Omega u^{n+1}v+\Delta t\int_\Omega\nabla u^{n+1}\cdot\nabla v
=\int_\Omega (u^n+\Delta t f^{n+1})v.
$$


In [ ]:
banner("4. Time-dependent heat equation")
domain = make_cube(12, "heat cube")
plot_mesh_surface_edges(domain, "Heat cube mesh: 3D surface with edges")
save_mesh_vtu(domain, "heat_mesh.vtu")

V = scalar_space(domain, 1)
problem_start("Heat", domain, V)
x = ufl.SpatialCoordinate(domain)
t = fem.Constant(domain, PETSc.ScalarType(0.0))
dt = 0.02
T = 0.20
num_steps = int(round(T / dt))

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
u_n = fem.Function(V, name="u_n")
u_h = fem.Function(V, name="u_heat")
f = ufl.sin(np.pi * x[0]) * ufl.sin(np.pi * x[1]) * ufl.sin(np.pi * x[2]) * ufl.exp(-t)

facets = cube_facets(domain, all_boundary)
dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, facets)
bc = fem.dirichletbc(PETSc.ScalarType(0.0), dofs, V)

a = (u * v + dt * ufl.dot(ufl.grad(u), ufl.grad(v))) * ufl.dx
L = (u_n + dt * f) * v * ufl.dx
A_heat = assemble_form_matrix(a, [bc])
problem = fem_petsc.LinearProblem(a, L, bcs=[bc], u=u_h, petsc_options=direct_lu_options())

snapshots = {}
energy = []
t0 = time.perf_counter()
log("[Heat] time loop     : start")
for nstep in range(1, num_steps + 1):
    t.value = PETSc.ScalarType(nstep * dt)
    step_t0 = time.perf_counter()
    problem.solve()
    step_elapsed = time.perf_counter() - step_t0
    u_h.x.scatter_forward()
    E = fem.assemble_scalar(fem.form(u_h**2 * ufl.dx))
    energy.append(comm.allreduce(E, op=MPI.SUM))
    if nstep in {1, num_steps // 2, num_steps}:
        snap = fem.Function(V, name=f"u_t_{float(t.value):.2f}")
        snap.x.array[:] = u_h.x.array
        snapshots[float(t.value)] = snap
    u_n.x.array[:] = u_h.x.array
    log(f"[Heat] step {nstep:02d}/{num_steps}  t={float(t.value):.2f}  energy={energy[-1]: .6e}  solve={step_elapsed:.4f}s")

solve_time = time.perf_counter() - t0
log("[Heat] time loop     : done")
report_ksp(problem, "Heat")
log(f"[Heat] solve timing : {solve_time:.4f} s")

save_history_csv("heat_energy_history.csv", ["step", "time", "energy"], ((i + 1, (i + 1) * dt, e) for i, e in enumerate(energy)))
heat_animation_path = export_heat_slice_animation(snapshots)
save_scalar_vtu(u_h, "heat_final_solution.vtu", "u")
save_scalar_slice_vtp(u_h, "heat_final_solution_zslice.vtp", "u")
for time_value, snap in snapshots.items():
    save_scalar_vtu(snap, f"heat_solution_t_{time_value:.2f}.vtu", "u")
    save_scalar_slice_vtp(snap, f"heat_solution_t_{time_value:.2f}_zslice.vtp", "u")
plot_scalar_surface_edges(u_h, "Heat final solution on 3D cube surface", "u")
for time_value, snap in snapshots.items():
    plot_scalar_slice(snap, f"Heat solution at t={time_value:.2f}, z=0.5 slice", "u")
plot_history(energy, "Heat energy history", r"$\int_\Omega u_h^2\,dx$")


In [ ]:
plot_sparsity(A_heat, "Heat implicit-Euler matrix")


## 5. Helmholtz equation with periodic MPC constraints

$$
-\Delta u-k^2u=f,\qquad k=10,\qquad
f=\sin(2\pi x)\cos(2\pi y)\sin(\pi z).
$$

Periodic constraints:

$$
u(0,y,z)=u(1,y,z),\qquad u(x,0,z)=u(x,1,z),
$$

with $u=0$ on $z=0,1$.

Weak form:

$$
\int_\Omega\nabla u\cdot\nabla v-k^2\int_\Omega uv=\int_\Omega fv.
$$

MPC eliminates slave DOFs by replacing them with master-DOF combinations before the PETSc solve.


In [ ]:
banner("5. Helmholtz equation with periodic MPC constraints")
if dolfinx_mpc is None:
    raise ImportError("dolfinx_mpc is required for the MPC Helmholtz example.")

domain = make_cube(8, "Helmholtz MPC cube")
plot_mesh_surface_edges(domain, "Helmholtz MPC cube mesh: 3D surface with edges")
save_mesh_vtu(domain, "helmholtz_mpc_mesh.vtu")

V = scalar_space(domain, 1)
problem_start("Helmholtz MPC", domain, V)
x = ufl.SpatialCoordinate(domain)
k = fem.Constant(domain, PETSc.ScalarType(10.0))
f = ufl.sin(2 * np.pi * x[0]) * ufl.cos(2 * np.pi * x[1]) * ufl.sin(np.pi * x[2])

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
a = (ufl.dot(ufl.grad(u), ufl.grad(v)) - k**2 * u * v) * ufl.dx
L = f * v * ufl.dx

z_facets = cube_facets(domain, lambda X: np.isclose(X[2], 0.0) | np.isclose(X[2], 1.0))
z_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, z_facets)
bc_z = fem.dirichletbc(PETSc.ScalarType(0.0), z_dofs, V)
bcs = [bc_z]

mpc = dolfinx_mpc.MultiPointConstraint(V)


# Select slave DOFs on x=1, excluding Dirichlet planes.
def slave_x1(X):
    return np.isclose(X[0], 1.0) & (~np.isclose(X[2], 0.0)) & (~np.isclose(X[2], 1.0))


# Map x=1 slave coordinates to x=0 master coordinates.
def map_x1_to_x0(X):
    Y = X.copy()
    Y[0] -= 1.0
    return Y


# Select slave DOFs on y=1, avoiding already constrained x=1 edge and Dirichlet planes.
def slave_y1(X):
    return (
        np.isclose(X[1], 1.0)
        & (~np.isclose(X[0], 1.0))
        & (~np.isclose(X[2], 0.0))
        & (~np.isclose(X[2], 1.0))
    )


# Map y=1 slave coordinates to y=0 master coordinates.
def map_y1_to_y0(X):
    Y = X.copy()
    Y[1] -= 1.0
    return Y


log("[Helmholtz MPC] constraints   : build periodic x and y maps")
mpc.create_periodic_constraint_geometrical(V, slave_x1, map_x1_to_x0, bcs)
mpc.create_periodic_constraint_geometrical(V, slave_y1, map_y1_to_y0, bcs)
mpc.finalize()
log("[Helmholtz MPC] constraints   : finalized")

A_helmholtz_mpc = assemble_mpc_matrix(a, mpc, bcs)
problem = dolfinx_mpc.LinearProblem(
    a,
    L,
    mpc,
    bcs=bcs,
    petsc_options=gmres_lu_options(),
)
uh, solve_time = timed_solve("Helmholtz MPC", problem.solve)
uh.name = "u_helmholtz_mpc"
mpc.backsubstitution(uh)
uh.x.scatter_forward()
report_ksp(problem, "Helmholtz MPC")


# Measure periodic mismatch between paired boundary DOFs.
def periodic_pair_norm(u_h, axis):
    coords = u_h.function_space.tabulate_dof_coordinates()
    values = np.asarray(u_h.x.array.real)
    lo = {}
    hi = {}
    other_axes = [i for i in range(3) if i != axis]
    for dof, X in enumerate(coords):
        if np.isclose(X[2], 0.0) or np.isclose(X[2], 1.0):
            continue
        key = tuple(np.round(X[other_axes], 12))
        if np.isclose(X[axis], 0.0):
            lo[key] = values[dof]
        elif np.isclose(X[axis], 1.0):
            hi[key] = values[dof]
    diffs = [hi[key] - lo[key] for key in hi.keys() if key in lo]
    local = float(np.dot(diffs, diffs)) if diffs else 0.0
    return np.sqrt(comm.allreduce(local, op=MPI.SUM)), len(diffs)


norm_x, pairs_x = periodic_pair_norm(uh, axis=0)
norm_y, pairs_y = periodic_pair_norm(uh, axis=1)
log(f"[Helmholtz MPC] periodic x  : norm={norm_x:.6e}, pairs={pairs_x}")
log(f"[Helmholtz MPC] periodic y  : norm={norm_y:.6e}, pairs={pairs_y}")
log(f"[Helmholtz MPC] solve timing: {solve_time:.4f} s")
save_history_csv("helmholtz_periodicity_residuals.csv", ["axis", "norm", "pairs"], [(0, norm_x, pairs_x), (1, norm_y, pairs_y)])
save_scalar_vtu(uh, "helmholtz_mpc_solution.vtu", "u")
save_scalar_slice_vtp(uh, "helmholtz_mpc_solution_zslice.vtp", "u")

plot_scalar_surface_edges(uh, "Helmholtz MPC solution on 3D cube surface", "u", cmap="coolwarm")
plot_scalar_slice(uh, "Helmholtz MPC solution, z=0.5 slice", "u", cmap="coolwarm")


In [ ]:
plot_sparsity(A_helmholtz_mpc, "Helmholtz matrix after MPC constraints")
